# Saving your network weights

For every question in the assessment you are required to submit your saved network weights along with a Python file that can be used to load your network and then run some test data through the network. This notebook sketches out what you need to do. Let's use Q2 (Where's Wally?) as an example, for which you need to submit a file `predict_wally.py`.

If your trained network is stored in `model` then saving the network weights is as simple as:

In [ ]:
torch.save(model.state_dict(), 'q2_weights.pkl')

In your test script for the assessment, you need to load the network weights. First, define the network as usual (make sure that you haven't changed anything about the structure between training and saving versus reloading it) and instantiate it (meaning it will initially have random weights). Assuming `model` is the network object then load the weights using:

In [ ]:
model.load_state_dict(torch.load('q2_weights.pkl',map_location=torch.device(device)))

You can assume that the weight file is in the current folder. Give each question's weights a different file name (for example `q1_weights.pkl`, ..., `q4_weights.pkl`), because I will run all of your scripts from the same folder. The size limit in the question applies to this file.

# Test script

A skeleton template for `predict_wally.py` is attached to "Model and test script submission template" within the Assessment section of the VLE.

The tile below checks that your script works in the way I will call it. Start a fresh runtime and upload to the session storage of this notebook:

* your `predict_wally.py`,
* your weights file `q2_weights.pkl`,
* `q2_val.npz` from `q2_data.zip`.

The tile loads the first 50 validation scenes, decodes them exactly as I will decode the test scenes, and passes them to your `predict` function as a $B\times 3\times 256\times 256$ tensor with values in $(0,1)$. It then checks the shape of your output and computes the AP on these 50 scenes. (This is only a check that everything runs: the AP on 50 scenes is very noisy, and the test set is much larger.) Set `LEVEL` to `'H'` or `'M'` to match your paper.

In [ ]:
import io, os
import numpy as np
import torch
from PIL import Image
from shapely.geometry import Polygon
import predict_wally

LEVEL = 'H'   # 'H' or 'M'
B = 50

# Load and decode the first B validation scenes (see README_Q2.md)
d = np.load('q2_val.npz')
ends = np.cumsum(d['lengths'])
starts = ends - d['lengths']
scenes = np.stack([np.asarray(Image.open(io.BytesIO(d['jpeg'][s:e].tobytes())).convert('RGB'))
                   for s, e in zip(starts[:B], ends[:B])])           # B x 256 x 256 x 3, uint8
targets = d['targets'][:B]                                          # B x 5 (H) or B x 6 (M)

images = torch.from_numpy(scenes).permute(0, 3, 1, 2).float() / 255
images = images.clamp(1/512, 1 - 1/512)                             # values in (0,1)

detections = predict_wally.predict(images)
assert detections.shape == (B, 5 if LEVEL == 'H' else 6)
detections = detections.float().cpu().numpy()

def corners(cx, cy, w, h, theta=0.0):
    t = np.deg2rad(theta)
    R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    c = np.array([[-w/2, -h/2], [w/2, -h/2], [w/2, h/2], [-w/2, h/2]])
    return c @ R.T + [cx, cy]

def iou(a, b):
    pa, pb = Polygon(corners(*a)), Polygon(corners(*b))
    inter = pa.intersection(pb).area
    return inter / (pa.area + pb.area - inter)

def angle_error(a, b):
    e = abs(a - b) % 360
    return min(e, 360 - e)

# A scene is a true positive if my face is present and your box (and angle, at M-level) is correct
present = targets[:, 0] > 0
correct = np.zeros(B, bool)
for i in np.nonzero(present)[0]:
    correct[i] = iou(detections[i, 1:], targets[i, 1:]) >= 0.5
    if LEVEL == 'M':
        correct[i] &= angle_error(detections[i, 5], targets[i, 5]) <= 20

# Average precision over scenes ranked by your score p
order = np.argsort(-detections[:, 0])
tp = correct[order]
precision = np.cumsum(tp) / np.arange(1, B + 1)
AP = (precision * tp).sum() / present.sum()
print(f'AP on {B} validation scenes: {AP:.3f}')

print(f'Weights file size: {os.path.getsize("q2_weights.pkl") / 2**20:.2f} MiB (limit 4 MiB)')

Check that if you upload your `predict_wally.py`, your weights and `q2_val.npz` to the session storage of this notebook then the above tile runs without error and gives a sensible AP.

The same pattern applies to the other questions: Q1 (`predict_shapes.py`), Q3 (`predict_scenes.py`) and Q4 (`complete_hands.py`). Refer to the assessment paper for the exact function signatures and input/output formats for each question, and to each question's README for how to load its validation data.